# Fine-Tuning ResNet50 for Skin Cancer Classification

This notebook demonstrates transfer learning using the pre-trained ResNet50 model for skin cancer classification on the HAM10000 dataset.

## Import Libraries

In [ ]:
import os
from glob import glob

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from keras.utils.np_utils import to_categorical

from keras.preprocessing.image import ImageDataGenerator
from keras import layers
from keras import Model
from keras.applications.resnet50 import ResNet50
from keras.optimizers import Adam
from keras.callbacks import ReduceLROnPlateau

import matplotlib.pyplot as plt

## Load the Dataset

In [ ]:
X_train = np.load("/kaggle/input/skin-cancer-dataset/256_192_train.npy")
y_train = np.load("/kaggle/input/skin-cancer-dataset/train_labels.npy")

X_val = np.load("/kaggle/input/skin-cancer-dataset/256_192_val.npy")
y_val = np.load("/kaggle/input/skin-cancer-dataset/val_labels.npy")

In [ ]:
X_train.shape, X_val.shape

In [ ]:
y_train = to_categorical(y_train, num_classes=7)
y_val = to_categorical(y_val, num_classes=7)

## Load Pre-trained ResNet50

In [ ]:
pre_trained_model = ResNet50(input_shape=(192, 256, 3), include_top=False, weights='imagenet')

for layer in pre_trained_model.layers:
    layer.trainable = False

print(len(pre_trained_model.layers))

## Build Custom Classification Head

In [ ]:
last_output = pre_trained_model.output

x = layers.GlobalAveragePooling2D()(last_output)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(7, activation='softmax')(x)

model = Model(inputs=pre_trained_model.input, outputs=x)

optimizer = Adam(lr=0.0001, beta_1=0.9, beta_2=0.999)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

model.summary()

## Data Augmentation

In [ ]:
train_datagen = ImageDataGenerator(
    rotation_range=60,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

train_datagen.fit(X_train)

val_datagen = ImageDataGenerator()
val_datagen.fit(X_val)

## Phase 1: Feature Extraction

In [ ]:
batch_size = 32
epochs = 5

history_phase1 = model.fit_generator(
    train_datagen.flow(X_train, y_train, batch_size=batch_size),
    epochs=epochs,
    validation_data=val_datagen.flow(X_val, y_val),
    verbose=1,
    steps_per_epoch=X_train.shape[0] // batch_size,
    validation_steps=X_val.shape[0] // batch_size
)

## Phase 2: Fine-Tuning

In [ ]:
for layer in pre_trained_model.layers[-30:]:
    layer.trainable = True

optimizer = Adam(lr=0.00001, beta_1=0.9, beta_2=0.999)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['acc'])

learning_rate_reduction = ReduceLROnPlateau(
    monitor='val_acc',
    patience=3,
    verbose=1,
    factor=0.5,
    min_lr=0.000001,
    cooldown=2
)

In [ ]:
batch_size = 32
epochs = 25

history_phase2 = model.fit_generator(
    train_datagen.flow(X_train, y_train, batch_size=batch_size),
    epochs=epochs,
    validation_data=val_datagen.flow(X_val, y_val),
    verbose=1,
    steps_per_epoch=X_train.shape[0] // batch_size,
    validation_steps=X_val.shape[0] // batch_size,
    callbacks=[learning_rate_reduction]
)

## Validation Evaluation

In [ ]:
loss_val, acc_val = model.evaluate(X_val, y_val, verbose=1)
print(f"Validation: accuracy = {acc_val:.4f}  ;  loss = {loss_val:.4f}")

## Testing

In [ ]:
X_test = np.load("/kaggle/input/skin-cancer-dataset/256_192_test.npy")
y_test = np.load("/kaggle/input/skin-cancer-dataset/test_labels.npy")
y_test = to_categorical(y_test, num_classes=7)

loss_test, acc_test = model.evaluate(X_test, y_test, verbose=1)
print(f"Test: accuracy = {acc_test:.4f}  ;  loss = {loss_test:.4f}")

## Save Model

In [ ]:
model.save("/kaggle/working/ResNet50.h5")
print("Model saved to /kaggle/working/ResNet50.h5")

## Visualize Training

In [ ]:
acc = history_phase1.history.get('accuracy', history_phase1.history.get('acc', []))
acc += history_phase2.history.get('accuracy', history_phase2.history.get('acc', []))

val_acc = history_phase1.history.get('val_accuracy', history_phase1.history.get('val_acc', []))
val_acc += history_phase2.history.get('val_accuracy', history_phase2.history.get('val_acc', []))

loss = history_phase1.history['loss'] + history_phase2.history['loss']
val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training')
plt.plot(epochs_range, val_acc, label='Validation')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training')
plt.plot(epochs_range, val_loss, label='Validation')
plt.legend()
plt.title('Loss')

plt.show()